# Lab 03 — Logistic Regression — Telco Churn
**Classification Track** · Intermediate · ~60 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Build a binary classifier with logistic regression
2. Encode categoricals and handle blank TotalCharges
3. Tune decision threshold for precision/recall trade-off
4. Read coefficients as log-odds for interpretability

## Datasets (this folder)
- `Telco-Customer-Churn.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-03-logistic-regression-churn/lab-03-logistic-regression-churn.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Paste the lab token from the ViLabs page into `VL_TOKEN` in **Cell 0**, then run that cell first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Do not modify the validation function. Write your code in the exercise cells, run those cells to verify, then run the validation cell.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`Telco-Customer-Churn.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-03-logistic-regression-churn"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-03-logistic-regression-churn/bundle/dataset.zip"
NEED = ["Telco-Customer-Churn.csv"]  # unzipped files used by the code below


import json as _json
import urllib.parse as _urlparse
import urllib.request as _urlrequest

VLABS_API = "https://prod-api.vilabs.app"
VL_TOKEN = ""  # paste the lab token shown on the ViLabs page after Start Lab

def vlabs_submit():
    """Validation function. Do not modify."""
    import inspect
    token = str(VL_TOKEN or "").strip()
    if not token:
        print("ViLabs: paste the lab token into VL_TOKEN in Cell 0, then re-run this cell.")
        return None
    names = ["churn_rate", "recall_at_best"]
    answers = {}
    missing = []
    for name in names:
        fn = globals().get(name)
        if not callable(fn):
            missing.append(name)
            continue
        answers[name] = inspect.getsource(fn)
    if missing:
        print("ViLabs: write these functions before validating:", ", ".join(missing))
        return None
    base = VLABS_API.rstrip("/") + "/colab/labs/" + LAB_ID + "/session"
    ua = {"User-Agent": "Mozilla/5.0 (compatible; ViLabs-Colab/1.0)"}

    def _read(req):
        try:
            with _urlrequest.urlopen(req, timeout=90) as resp:
                return _json.loads(resp.read().decode())
        except _urlrequest.HTTPError as exc:
            print("ViLabs HTTP", exc.code, exc.read().decode(errors="replace")[:300])
            return None

    payload = _json.dumps({"token": token, "answers": answers}).encode()
    result = _read(_urlrequest.Request(
        base + "/validate/",
        data=payload,
        headers={"Content-Type": "application/json", **ua},
    ))
    if result is None:
        return None
    print("ViLabs validated", result.get("validated"))
    for row in result.get("variables") or []:
        print(row.get("name"), "pass" if row.get("ok") else "fail")
    return result

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Telecom: Predicting Customer Churn

> **Scenario:** You are a data scientist at a telecom company. The marketing team wants to identify customers likely to churn (leave) so they can proactively offer retention incentives. You have 7,043 customer records with 20 features and a binary `Churn` target. Your task is to build a logistic regression classifier, interpret its coefficients, and tune the decision threshold for business needs.
>
> **You will learn:** binary classification, categorical encoding, logistic regression coefficients as log-odds, confusion matrix, precision/recall, threshold tuning.
> **Time:** ~60 minutes. **Level:** Intermediate. **Needs:** pandas + sklearn + matplotlib. **Env:** 🟢 Colab only.

### Mental Map

| Concept | What it means | Where it shows up |
|---|---|---|
| Binary classification | Predict one of two classes (churn / no churn) | `LogisticRegression()` |
| Log-odds | Log of the odds ratio; coefficients are additive on this scale | `model.coef_` |
| Sigmoid | Maps log-odds to probability (0–1) | `model.predict_proba()` |
| Decision threshold | Probability cutoff for classifying as positive | Default 0.5 |
| Confusion matrix | TP, FP, TN, FN counts | `confusion_matrix()` |
| Precision | Of predicted churners, how many actually churned | `precision_score()` |
| Recall | Of actual churners, how many did we catch | `recall_score()` |

---

## 1. Load and Explore the Data

Churn prediction is one of the most valuable applications of machine learning in business. When a customer leaves, you lose not just their current revenue but their entire future lifetime value — and acquiring a new customer costs 5-25× more than retaining an existing one. A good churn model lets you intervene before it is too late: offer a discount, assign a dedicated account manager, or simply reach out to understand what is wrong.

The Telco Customer Churn dataset is a classic benchmark. It contains 7,043 customers, each described by 20 features: demographics (gender, age, whether they have a partner or dependents), account information (tenure, contract type, billing method), and services subscribed (internet type, online security, tech support, streaming). The target is simple: did the customer churn (Yes/No)?

The first thing you notice is the **class imbalance**: only 26.5% of customers churned. This is realistic — most customers stay — but it creates a trap. A model that always predicts "no churn" achieves 73.5% accuracy while being completely useless. Every metric and every design choice in this lab must account for this imbalance.

The second thing you notice is a data quality issue: `TotalCharges` is stored as text, not numbers. This is because 11 new customers have a blank value — they have not been billed yet. We will need to handle this before modelling.

Let us load the data and see the imbalance for ourselves.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"Telco-Customer-Churn.csv")
print(df.shape)
print(df.head())
print(df.dtypes)
print(df["Churn"].value_counts())
print(f"Churn rate: {(df['Churn'] == 'Yes').mean():.4f}")


**What to notice:**
- 7,043 rows, 21 columns (20 features + 1 target).
- The dataset is imbalanced: ~26.5% churn vs ~73.5% no-churn.
- `TotalCharges` is stored as object (string) because of blank values — needs cleaning.

> **Pitfall:** Class imbalance means accuracy alone is misleading. A model that always predicts "no churn" gets ~73.5% accuracy but is useless.

---

## 2. Clean and Encode the Data

Real-world data is messy, and this dataset is no exception. The `TotalCharges` column contains blank strings for customers who have not been billed yet — typically new customers with zero tenure. If we try to convert this column to numeric directly, pandas will raise an error. The fix is `pd.to_numeric(errors="coerce")`, which converts what it can and turns the rest into NaN. We then fill those NaNs with 0 — a reasonable assumption for customers who have not been charged.

Next, we encode the categorical variables. Most of the features are strings: "Yes"/"No" for binary options, "Month-to-month"/"One year"/"Two year" for contract types, "Electronic check"/"Mailed check"/etc. for payment methods. Machine learning models need numbers, so we use **one-hot encoding**: each category becomes its own binary column. A customer with a "One year" contract gets a 1 in the `Contract_One year` column and 0 in all other contract columns.

We use `drop_first=True` to avoid the dummy variable trap — if we kept all categories, the columns would be perfectly collinear (knowing all but one tells you the last), which can cause numerical instability in some models. Dropping the first category as a reference level solves this.

The result is a feature matrix with ~30 columns, all numeric, ready for modelling. The target is mapped to binary: 1 for churn, 0 for no churn. Let us build it.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop(columns=["Churn"])
y = df["Churn"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"Missing values: {X.isnull().sum().sum()}")


**What to notice:**
- After one-hot encoding, the feature matrix expands to ~30+ columns.
- The target is now binary: 1 = churn, 0 = no churn.
- No missing values remain after cleaning.

> **Pitfall:** `pd.to_numeric(errors="coerce")` turns blanks into NaN. Always fill them before modeling.

---

## 3. Fit Logistic Regression

Logistic regression is the workhorse of binary classification. Despite its name, it is a classification algorithm, not a regression algorithm. The "regression" part comes from the fact that it models the **log-odds** of the positive class as a linear combination of features, then squashes the result through a sigmoid function to get a probability.

Here is the intuition. For each customer, the model computes a weighted sum of their features: `z = w₁·tenure + w₂·MonthlyCharges + w₃·Contract_OneYear + ... + b`. This sum can be any number from negative to positive infinity. The sigmoid function `σ(z) = 1 / (1 + e^(-z))` maps this to a probability between 0 and 1. If the probability is above 0.5, we predict "churn"; otherwise, "no churn."

The coefficients tell a story. A positive coefficient means that feature increases the log-odds of churn; a negative coefficient means it decreases them. The magnitude tells you how strong the effect is. But remember: coefficients are only comparable when features are on similar scales. A coefficient of 0.5 for `tenure` (range 1-72) means something very different from 0.5 for `MonthlyCharges` (range 18-118).

We use `stratify=y` in the train/test split to ensure both sets have the same 26.5% churn rate. Without stratification, we might get a test set with 30% churn or 20% churn by chance, making our metrics unreliable.

The `max_iter=1000` parameter tells sklearn to allow up to 1000 iterations for the optimiser to converge. The default of 100 is often not enough for logistic regression on real-world data, and you will see a warning if it does not converge.

Let us fit the model and see how it performs.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

df = pd.read_csv(r"Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])
y = df["Churn"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")


**What to notice:**
- Accuracy may look decent (~78–80%) but is inflated by class imbalance.
- Precision and recall give a clearer picture of model performance on the minority class.
- The confusion matrix shows the exact TP/FP/TN/FN breakdown.

> **Pitfall:** Use `stratify=y` in `train_test_split` to preserve the class ratio in both train and test sets.

---

## 4. Interpret Coefficients as Log-Odds

A model you cannot explain is a model you cannot trust. In regulated industries — banking, insurance, healthcare — you may be legally required to explain why a model made a particular decision. Logistic regression is one of the few models that is both accurate and interpretable, which is why it remains the default choice for many business applications.

The key to interpreting logistic regression is understanding **odds ratios**. The coefficient for a feature tells you how much the log-odds change when that feature increases by one unit. But log-odds are hard to interpret intuitively. The odds ratio — `exp(coefficient)` — is much easier: it tells you how the *odds* of churn multiply when the feature increases by one unit.

For example, if the coefficient for `tenure` is -0.05, the odds ratio is `exp(-0.05) ≈ 0.95`. This means that for each additional month of tenure, the odds of churn multiply by 0.95 — a 5% decrease. A customer who has been with the company for 24 months has `0.95^24 ≈ 0.30` times the odds of churning compared to a new customer. That is a powerful retention story.

Conversely, if `InternetService_Fiber optic` has a coefficient of +0.8, the odds ratio is `exp(0.8) ≈ 2.23`. Fiber customers have 2.23 times the odds of churning compared to the reference category (DSL), all else being equal. This might indicate that fiber service has quality issues or that fiber customers are more price-sensitive.

Let us extract the coefficients and see which features drive churn.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

df = pd.read_csv(r"Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])
y = df["Churn"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": model.coef_[0],
    "odds_ratio": np.exp(model.coef_[0]),
}).sort_values("coef", ascending=False)

print("Top 5 churn INCREASERS:")
print(coef_df.head(5).to_string(index=False))
print("\nTop 5 churn DECREASERS:")
print(coef_df.tail(5).to_string(index=False))


**What to notice:**
- `tenure` (negative coefficient) means longer-tenured customers are less likely to churn.
- `Contract_One year` or `Contract_Two year` (negative) means longer contracts reduce churn.
- `InternetService_Fiber optic` (positive) may indicate fiber customers churn more.
- Odds ratio > 1 means increased churn odds; < 1 means decreased.

> **Pitfall:** Coefficients are only comparable when features are on similar scales. Standardize features if you want to compare magnitudes directly.

---

## 5. Threshold Tuning

Here is a question that trips up even experienced practitioners: what probability cutoff should we use to classify a customer as "likely to churn"? The default is 0.5 — if the model says the probability of churn is above 50%, we predict churn. But this default is almost never optimal for imbalanced data.

Think about the business context. If we predict a customer will churn, the retention team will intervene — perhaps offering a discount, a free month, or a dedicated call. Each intervention has a cost. If we intervene on too many customers, we waste money on retention offers for people who would have stayed anyway. If we intervene on too few, we miss customers we could have saved.

The **precision-recall trade-off** captures this tension. Lowering the threshold means we flag more customers as likely to churn. This increases **recall** (we catch more actual churners) but decreases **precision** (more of our flagged customers would have stayed anyway). Raising the threshold does the opposite.

The right threshold depends on the relative costs. If a retention offer costs $10 and a lost customer is worth $500, you want high recall — flag anyone with even a 2% chance of churning. If the offer costs $100 and the customer is worth $200, you want high precision — only flag customers who are very likely to leave.

Let us sweep through different thresholds and see how precision and recall change.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

df = pd.read_csv(r"Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])
y = df["Churn"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)
results = []
for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    results.append({
        "threshold": round(t, 2),
        "precision": precision_score(y_test, y_pred_t),
        "recall": recall_score(y_test, y_pred_t),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

target_recall = 0.80
valid = results_df[results_df["recall"] >= target_recall]
best_threshold = valid["threshold"].max()
print(f"\nHighest threshold with recall >= {target_recall}: {best_threshold}")


**What to notice:**
- Lowering the threshold increases recall (catch more churners) but decreases precision (more false alarms).
- The business must decide the acceptable trade-off.
- A threshold around 0.3–0.4 often achieves recall ≥ 0.80.

> **Pitfall:** Don't tune the threshold on the test set in production — use a validation set or cross-validation to avoid overfitting the threshold.

---

## Exercises

Do not modify the validation function. Write your code in the exercise cells, run those cells to verify, then run the validation cell.

### Exercise 1 — Churn Base Rate
What percentage of customers in the dataset churned?


**Follow-up:** Why is the base rate important for evaluating classifier performance?

### Exercise 2 — Precision and Recall at Threshold 0.5
Compute precision and recall using the default 0.5 threshold.


**Follow-up:** Which metric matters more for a retention campaign — precision or recall?

### Exercise 3 — Threshold for Recall ≥ 0.80
Find the highest threshold that achieves recall of at least 0.80.


**Follow-up:** What happens to precision at this threshold?

<details>
<summary>Hint</summary>


```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

df = pd.read_csv(r"Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])
y = df["Churn"]
cat_cols = X.select_dtypes(include="object").columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]

# Exercise 1
print(f"Churn rate: {y.mean():.4f}")
print(f"Churn count: {y.sum()} / {len(y)}")

# Exercise 2
y_pred = (y_proba >= 0.5).astype(int)
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")

# Exercise 3
for t in np.arange(0.1, 0.9, 0.05):
    y_pred_t = (y_proba >= t).astype(int)
    r = recall_score(y_test, y_pred_t)
    if r >= 0.80:
        print(f"Threshold {t:.2f}: recall = {r:.4f}")
```


</details>

---


In [ ]:
def churn_rate():
    """churn rate."""
    pass


In [ ]:
def recall_at_best():
    """What happens to precision at this threshold?"""
    pass


In [ ]:
vlabs_submit()


### What to learn next
- Handling class imbalance with SMOTE or class weights
- ROC curves and AUC for threshold-independent evaluation
- Gradient boosting classifiers (XGBoost) for churn prediction

*Files in this folder: Telco-Customer-Churn.csv.*


---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
